<a href="https://colab.research.google.com/github/Nathi774-dev/Nathi774-dev/blob/AI%2FML/microsoft_projects/Hotel_Reviewer_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### The filtered dataset is used here

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import pandas as pd

In [56]:
df = pd.read_csv("drive/MyDrive/Documents/filtered_Hotel_Reviews.csv")
df

,Unnamed: 0,Hotel_Address,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,...,Calc_Average_Score,Average_Score_Difference,Leisure_Trip,Couple,Solo_Trip,Business_Trip,Group,Family_with_young_children,Family_with_older_children,With_a_pet
0,495945,"Milan, Italy",7/19/2017,8.3,Best Western Hotel Astoria,Switzerland,The rooms a smaller than usual 4 hotels,10,1,All the staff were really helpful and friendl...,...,8.46,-0.76,0,0,0,0,0,0,1,0
1,43688,"Paris, France",5/31/2017,7.9,Mercure Paris Porte d Orleans,Luxembourg,A bit on the expensive side not necessarily o...,38,1,The hotel was clean and the check in and chec...,...,8.18,-0.68,0,1,0,0,0,0,1,0
2,178253,"Paris, France",5/22/2017,9.6,Renaissance Paris Vendome Hotel,United States of America,The restaurant in the front corner of the hot...,25,1,The staff was very friendly The Concierge was...,...,8.58,-0.68,0,1,0,0,0,0,1,0
3,111027,"Paris, France",3/5/2017,10.0,Hotel Stendhal Place Vend me Paris MGallery by...,Australia,Nothing really I just wish European hotels ha...,17,1,Great hotel Typical Accor management being pa...,...,9.47,-0.67,0,0,0,0,0,0,1,0
4,257749,"Paris, France",7/5/2017,9.0,Hotel De Vigny,United States of America,My suite was stunning once upon a time The fu...,87,1,The staff was exceedingly friendly and helpfu...,...,8.04,-0.54,0,0,0,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1487,151416,"Paris, France",6/2/2017,5.8,Best Western Allegro Nation,Poland,issues with WiFi average quality of the room ...,33,1,fairly good price friendly staff,...,7.06,0.74,0,0,0,0,0,0,1,0
1488,22189,"Paris, France",8/3/2017,4.6,Holiday Inn Paris Montparnasse Pasteur,United Kingdom,The hotel needs an upgrade as bathroom was ve...,51,1,Staff were very pleasant to deal with,...,6.33,0.77,0,0,0,0,0,0,1,0
1489,250308,"Paris, France",2/17/2017,3.8,MARQUIS Faubourg St Honor Relais Ch teaux,United Kingdom,They have 2 members of staff working as manag...,377,1,Nothing positive,...,7.73,0.87,0,0,0,0,0,0,1,0
1490,68936,"Paris, France",6/23/2017,4.6,Villa Eugenie,Netherlands,Location is very old not invested in past yea...,41,1,Location is central accessible by train parki...,...,5.86,0.94,0,0,0,0,0,0,1,0


In [57]:
df.Tags = df.Tags.str.strip("[']")
df.Tags = df.Tags.str.replace(" ', '", ",", regex = False)

In [58]:
tag_list_df = df.Tags.str.split(',', expand=True)

df["Tag_1"] = tag_list_df[0].str.strip()
df["Tag_2"] = tag_list_df[1].str.strip()
df["Tag_3"] = tag_list_df[2].str.strip()
df["Tag_4"] = tag_list_df[3].str.strip()
df["Tag_5"] = tag_list_df[4].str.strip()
df["Tag_6"] = tag_list_df[5].str.strip()

In [59]:
df['Tag_6']

,Tag_6
0,None
1,None
2,None
3,None
4,None
...,...
1487,None
1488,None
1489,None
1490,None


In [60]:
df_tags = df.melt(value_vars=["Tag_1", "Tag_2", "Tag_3", "Tag_4", "Tag_5", "Tag_6"])

In [61]:
df_tags

,variable,value
0,Tag_1,Leisure trip
1,Tag_1,Leisure trip
2,Tag_1,Leisure trip
3,Tag_1,Business trip
4,Tag_1,Leisure trip
...,...,...
8947,Tag_6,None
8948,Tag_6,None
8949,Tag_6,None
8950,Tag_6,None


In [62]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df_tags)

https://docs.google.com/spreadsheets/d/1T3-2basKHCAQsImchFgY6z5OBhPbiAQUG2y74cBnnh4/edit#gid=0


In [63]:
tag_vc = df_tags.value.value_counts()
df_tags.shape

(8952, 2)

In [64]:
df_tags = df_tags[~df_tags.value.str.contains("Standard|room|Stayed|device|Beds|Suite|Studio|King|Superior|Double", na=False, case=False)]
tag_vc = df_tags.value.value_counts().reset_index(name="count").query("count < 1000")
print(tag_vc[:10])

                         value  count
1                       Couple    692
2   Family with young children    327
3                Solo traveler    238
4                        Group    224
5                Business trip    152
6       Travelers with friends     11
7                   With a pet      3
8             Family Apartment      2
9            Classic Apartment      1
10            Classic Designer      1


In [65]:
import nltk as nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [66]:
vader_sentiment = SentimentIntensityAnalyzer()

In [71]:
def calc_sentiment(review):
  if review == "No Negative" or review == "No Positive":
    return 0
  return vader_sentiment.polarity_scores(review)["compound"]

In [72]:
cache = set(stopwords.words("english"))
def remove_stopwords(review):
  text = " ".join([word for word in review.split() if word not in cache])
  return text

In [73]:
df.Negative_Review = df.Negative_Review.apply(remove_stopwords)
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)

In [74]:
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)

In [75]:
df["Negative_Sentiment"]

,Negative_Sentiment
0,0.0000
1,0.0258
2,0.8481
3,-0.3559
4,0.9095
...,...
1487,-0.2732
1488,0.5106
1489,0.7096
1490,0.1027


In [76]:
print(df[["Negative_Review", "Negative_Sentiment"]])

                                        Negative_Review  Negative_Sentiment
0                      The rooms smaller usual 4 hotels              0.0000
1     A bit expensive side necessarily room prices p...              0.0258
2     The restaurant front corner hotel great servic...              0.8481
3     Nothing really I wish European hotels laundry ...             -0.3559
4     My suite stunning upon time The furniture show...              0.9095
...                                                 ...                 ...
1487  issues WiFi average quality room considering 4...             -0.2732
1488  The hotel needs upgrade bathroom dated clean B...              0.5106
1489  They 2 members staff working management waiter...              0.7096
1490  Location old invested past years time hotel mu...              0.1027
1491  The hotel situated worst area Paris black cab ...             -0.8723

[1492 rows x 2 columns]


In [77]:
df = df.reindex(["Hotel_Name", "Hotel_Address", "Total_Number_of_Reviews", "Average_Score", "Reviewer_Score", "Negative_Sentiment", "Positive_Sentiment", "Reviewer_Nationality", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Group", "Family_with_young_children", "Family_with_older_children", "With_a_pet", "Negative_Review", "Positive_Review"], axis=1)
print("Saving files to the folder")
df.to_csv("drive/MyDrive/Documents/Hotel_Reviews_NLP.csv")

Saving files to the folder
